# Week 4: HealthConnect Project Kickoff & Problem Understanding
## Phase 1 & 2: Project Review and Initial Data Assessment

**The Business Problem:**
HealthConnect Clinic is experiencing operational inefficiencies due to patients missing scheduled appointments. My objective as a Data Scientist is to define a machine learning problem to predict appointment no-shows, allowing the clinic to optimize scheduling, reduce wasted clinical hours, and improve patient support.

**Initial Data Assessment:**
I am reviewing the `HealthConnect_Appointment_Data.csv` dataset to assess its quality, structure, and suitability for a predictive no-show model. I need to understand the variables available, check for missing values, and observe the baseline distribution of appointment outcomes before defining my exact modelling approach.

In [1]:
import pandas as pd
import numpy as np

file_name = 'HealthConnect_Appointment_Data.csv'
df = pd.read_csv(file_name)

In [2]:
print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")

Total Rows: 5000
Total Columns: 18


In [3]:
print("\n--- DATA TYPES & STRUCTURE ---")
df.info()

# Checking for Missing Values
print("\n--- MISSING VALUES ---")
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("No missing values found in the dataset.")


--- DATA TYPES & STRUCTURE ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   appointment_id         5000 non-null   object 
 1   patient_id             5000 non-null   object 
 2   gender                 5000 non-null   object 
 3   age                    5000 non-null   int64  
 4   age_group              5000 non-null   object 
 5   appointment_type       5000 non-null   object 
 6   booking_date           5000 non-null   object 
 7   appointment_date       5000 non-null   object 
 8   appointment_day        5000 non-null   object 
 9   appointment_time       5000 non-null   object 
 10  booking_lead_days      5000 non-null   int64  
 11  previous_appointments  5000 non-null   int64  
 12  previous_no_shows      5000 non-null   int64  
 13  reminder_sent          5000 non-null   object 
 14  reminder_channel       3

In [4]:
print("\n--- FIRST 5 ROWS ---")
display(df.head())


--- FIRST 5 ROWS ---


,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show


In [5]:
# Checking Target Variable Distribution
print("\n--- TARGET VARIABLE DISTRIBUTION (appointment_outcome) ---")
print(df['appointment_outcome'].value_counts())
print("\nPercentages:")
print((df['appointment_outcome'].value_counts(normalize=True) * 100).round(2))


--- TARGET VARIABLE DISTRIBUTION (appointment_outcome) ---
appointment_outcome
No-Show      2423
Attended     2314
Cancelled     263
Name: count, dtype: int64

Percentages:
appointment_outcome
No-Show      48.46
Attended     46.28
Cancelled     5.26
Name: proportion, dtype: float64


## Phase 2: Machine Learning Problem Definition & Target Variable Strategy

**The Machine Learning Problem:**
My objective is to develop a binary classification model that predicts the likelihood of a patient missing a scheduled appointment (a "No-Show") based on their profile and booking details. By predicting this risk at the exact time the appointment is booked, HealthConnect can proactively intervene (e.g., sending targeted SMS reminders or implementing smart overbooking) to reduce wasted clinical hours.

**Proposed Target Variable:**
The target variable is `appointment_outcome`.

**Handling Cancellations:**
The raw dataset contains three distinct outcomes: `Attended`, `No-Show`, and `Cancelled`.
*   A **Cancellation** is an active communication from the patient. It allows the clinic to free up the schedule and offer the slot to another patient. It is a fundamentally different behavior than a No-Show.
*   A **No-Show** is passive, unpredictable, and results in direct revenue/resource loss.

Therefore, to build a highly accurate predictive model, I will **filter out all rows where the outcome is `Cancelled`**. Including them would introduce noise and confuse the model. I will then binarize the remaining target variable for the classification algorithm:
*   **Class 1 (Positive Class):** `No-Show` (The event I am trying to predict).
*   **Class 0 (Negative Class):** `Attended` (The baseline successful outcome).

In [6]:
# === TARGET VARIABLE FILTERING & ENCODING ===

# Check the original shape and distribution
print(f"Original Dataset Shape: {df.shape}")
print("\nOriginal Target Distribution:")
print(df['appointment_outcome'].value_counts())

Original Dataset Shape: (5000, 18)

Original Target Distribution:
appointment_outcome
No-Show      2423
Attended     2314
Cancelled     263
Name: count, dtype: int64


In [7]:
# Filtering out 'Cancelled' appointments and keeping only 'Attended' and 'No-Show'
df_ml = df[df['appointment_outcome'] != 'Cancelled'].copy()

print(f"\nDataset Shape after dropping 'Cancelled': {df_ml.shape}")


Dataset Shape after dropping 'Cancelled': (4737, 18)


In [8]:
# Binarize the Target Variable
# No-Show = 1 (Positive Class), Attended = 0 (Negative Class)
df_ml['target_no_show'] = df_ml['appointment_outcome'].map({
    'No-Show': 1,
    'Attended': 0
})

# Verify the new binary target
print("\nNew Binary Target Distribution (target_no_show):")
print(df_ml['target_no_show'].value_counts())
print("\nPercentages:")
print((df_ml['target_no_show'].value_counts(normalize=True) * 100).round(2))

# Droping the original text column to avoid data leakage/confusion later
df_ml = df_ml.drop('appointment_outcome', axis=1)


New Binary Target Distribution (target_no_show):
target_no_show
1    2423
0    2314
Name: count, dtype: int64

Percentages:
target_no_show
1    51.15
0    48.85
Name: proportion, dtype: float64


## Phase 2b: Potential Input Features & Initial Modelling Approach

**Potential Input Features:**
Based on my review of the HealthConnect dataset, I have categorized the available variables into four logical groups that I plan to use as input features (X) for my predictive model:

1. **Patient Demographics:**
   * `age` and `age_group` (to capture life-stage mobility and health urgency).
   * `gender` (to check for any demographic variance in attendance).
2. **Appointment Logistics:**
   * `appointment_type` (e.g., General Consultation vs. Follow-up vs. Diagnostic Test).
   * `appointment_day` and `appointment_time` (Morning/Afternoon/Evening dynamics).
   * `booking_lead_days` (How far in advance the appointment was scheduled).
   * `distance_to_clinic_km` (Geographical friction and travel burden).
3. **Patient Reliability History:**
   * `previous_appointments` (Total historical engagement).
   * `previous_no_shows` (Historical unreliability).
4. **Clinic Interventions:**
   * `reminder_sent` (Whether an intervention was applied).
   * `reminder_channel` (SMS, WhatsApp, Email, or None).

*Note on `waiting_time_minutes`: I noticed missing values in this column during my initial assessment. Furthermore, if this represents the *actual* time spent waiting in the clinic, it is only known after the patient arrives, which would cause severe data leakage. I will investigate this further and likely exclude it from the final predictive features.*

**Initial Modelling Approach:**
For Week 5, I plan to frame this as a **Binary Classification** problem. Because "No-Shows" are typically the minority class in healthcare datasets, I anticipate a **class imbalance** issue.

To address this, my modelling strategy will include:
1. **Baseline Model:** Logistic Regression (to establish a baseline and provide interpretable odds ratios).
2. **Advanced Models:** Random Forest and XGBoost (to capture non-linear relationships, such as the interaction between `distance_to_clinic_km` and `reminder_channel`).
3. **Imbalance Handling:** I will experiment with SMOTE (Synthetic Minority Over-sampling Technique) and class weighting to ensure the model does not simply predict "Attended" every time.
4. **Evaluation Metrics:** Accuracy will be misleading here. I will prioritize **Recall** (to minimize missed No-Shows), **F1-Score**, and the **ROC-AUC** curve to evaluate my model's true predictive power.

In [9]:
# === FEATURE CATEGORIZATION & IMBALANCE CHECK ===

# Define my feature categories for documentation
demographics = ['age', 'age_group', 'gender']
logistics = ['appointment_type', 'appointment_day', 'appointment_time', 'booking_lead_days', 'distance_to_clinic_km']
history = ['previous_appointments', 'previous_no_shows']
interventions = ['reminder_sent', 'reminder_channel']

print("--- POTENTIAL INPUT FEATURES (X) ---")
print(f"Demographics ({len(demographics)}): {demographics}")
print(f"Logistics ({len(logistics)}): {logistics}")
print(f"History ({len(history)}): {history}")
print(f"Interventions ({len(interventions)}): {interventions}")



--- POTENTIAL INPUT FEATURES (X) ---
Demographics (3): ['age', 'age_group', 'gender']
Logistics (5): ['appointment_type', 'appointment_day', 'appointment_time', 'booking_lead_days', 'distance_to_clinic_km']
History (2): ['previous_appointments', 'previous_no_shows']
Interventions (2): ['reminder_sent', 'reminder_channel']


In [10]:
# Checking the exact class imbalance of my new binary target
print("\n--- TARGET CLASS IMBALANCE CHECK ---")
target_counts = df_ml['target_no_show'].value_counts()
total_records = len(df_ml)

print(f"Total Records for Modelling: {total_records}")
print(f"Attended (0): {target_counts[0]} ({(target_counts[0]/total_records)*100:.1f}%)")
print(f"No-Show (1): {target_counts[1]} ({(target_counts[1]/total_records)*100:.1f}%)")


--- TARGET CLASS IMBALANCE CHECK ---
Total Records for Modelling: 4737
Attended (0): 2314 (48.8%)
No-Show (1): 2423 (51.2%)


In [11]:
# Quick check on the suspicious 'waiting_time_minutes' column
print("\n--- INVESTIGATING 'waiting_time_minutes' ---")
missing_wait = df_ml['waiting_time_minutes'].isnull().sum()
print(f"Missing values in waiting_time_minutes: {missing_wait} ({(missing_wait/total_records)*100:.1f}%)")
print("Decision: Due to missing values and high risk of data leakage (actual wait time is only known if they show up), I will likely DROP this feature before Week 5 modelling.")


--- INVESTIGATING 'waiting_time_minutes' ---
Missing values in waiting_time_minutes: 58 (1.2%)
Decision: Due to missing values and high risk of data leakage (actual wait time is only known if they show up), I will likely DROP this feature before Week 5 modelling.


## Phase 3: Assumptions, Limitations, Risks & Dependencies

In defining my machine learning approach for predicting appointment no-shows, I have identified several critical assumptions, dataset limitations, and modelling risks that will directly impact my work in Week 5.

### 1. Key Assumptions
*   **Historical Reliability:** I assume that a patient's past behaviour (`previous_no_shows` and `previous_appointments`) is a strong, stable indicator of their future reliability.
*   **Reminder Efficacy:** I assume the `reminder_sent` and `reminder_channel` fields accurately reflect that the message was successfully delivered and received by the patient prior to the appointment time.
*   **Stationarity:** I assume that the underlying factors driving no-shows (e.g., traffic patterns, clinic operations) remain relatively consistent across the time period covered by the dataset.

### 2. Dataset Limitations
*   **Missing Geospatial & Operational Data:** The dataset contains limited missing values in `distance_to_clinic_km` and `waiting_time_minutes`. I will need to handle these via imputation or exclusion in Week 5. Furthermore, I lack external contextual data such as weather conditions, public transport strikes, or specific clinic staffing levels on the day of the appointment, which are known drivers of attendance.
*   **Lack of Clinical Urgency:** The `appointment_type` provides some context (e.g., Diagnostic Test vs. Follow-up), but I do not have data on the clinical urgency or severity of the patient's condition. A patient might miss a routine follow-up but would never miss an urgent oncology consultation.
*   **Anonymised Demographics:** The dataset only provides `age`, `age_group`, and `gender`. Socioeconomic factors (income, insurance type, employment status) are missing, yet they are often heavily correlated with healthcare access and attendance.

### 3. Risks, Dependencies & Ethical Considerations
*   **Data Leakage Risk (`waiting_time_minutes`):** There is a severe risk of data leakage with the `waiting_time_minutes` variable. If this represents the *actual* time the patient spent in the waiting room, this number only exists if the patient actually showed up. Including it as a predictive feature would cause the model to achieve artificially perfect accuracy during training but fail completely in production. I must verify the definition of this column and likely drop it before Week 5.
*   **Class Imbalance:** As observed in my initial assessment, "No-Shows" will be the minority class. If I do not handle this via techniques like SMOTE, class weighting, or threshold tuning, my model will simply predict "Attended" for every patient and achieve high accuracy but zero business value.
*   **Ethical Bias:** If my model learns that certain demographic groups (e.g., specific age groups or genders) have higher historical no-show rates, it might unfairly penalize them in future scheduling (e.g., systematically overbooking their slots or denying them prime appointment times). I must ensure my model evaluates *behavioural* and *logistical* features rather than relying heavily on protected demographic attributes.

In [12]:
# === INVESTIGATING DATA LEAKAGE RISK ===

# Check if waiting_time_minutes behaves differently for No-Shows vs Attended
# If a patient doesn't show up, they can't have an actual waiting time.
# If the dataset records 'NaN' or '0' for No-Shows, it is a massive data leakage signal.

print("--- Average Waiting Time by Appointment Outcome ---")
print(df.groupby('appointment_outcome')['waiting_time_minutes'].mean())

print("\n--- Missing Waiting Time Values by Outcome ---")
print(df.groupby('appointment_outcome')['waiting_time_minutes'].apply(lambda x: x.isnull().sum()))

print("\n--- Minimum Waiting Time by Outcome ---")
print(df.groupby('appointment_outcome')['waiting_time_minutes'].min())

--- Average Waiting Time by Appointment Outcome ---
appointment_outcome
Attended     24.289141
Cancelled    23.214559
No-Show      24.200754
Name: waiting_time_minutes, dtype: float64

--- Missing Waiting Time Values by Outcome ---
appointment_outcome
Attended     21
Cancelled     2
No-Show      37
Name: waiting_time_minutes, dtype: int64

--- Minimum Waiting Time by Outcome ---
appointment_outcome
Attended     2.0
Cancelled    2.0
No-Show      2.0
Name: waiting_time_minutes, dtype: float64
